In [ ]:
# ============================================================
# MODEL INVERSION ATTACK ON FASHION-MNIST
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 2. SELECT CPU OR GPU
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ------------------------------------------------------------
# 3. LOAD FASHION-MNIST DATASET
# ------------------------------------------------------------

print("\nLoading Fashion-MNIST dataset...")

transform = transforms.ToTensor()

train_data = torchvision.datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=128,
    shuffle=True
)


# ------------------------------------------------------------
# 4. DEFINE THE TARGET MODEL
# ------------------------------------------------------------

class SimpleModel(nn.Module):

    def __init__(self):
        super().__init__()

        # Convert 28 x 28 image into 784 values
        self.flatten = nn.Flatten()

        # First layer
        self.layer1 = nn.Linear(784, 256)

        # Activation function
        self.relu = nn.ReLU()

        # Output layer: 10 Fashion-MNIST classes
        self.layer2 = nn.Linear(256, 10)

    def forward(self, image):

        image = self.flatten(image)

        x = self.layer1(image)

        x = self.relu(x)

        output = self.layer2(x)

        return output


# Create the model
model = SimpleModel().to(device)


# ------------------------------------------------------------
# 5. TRAIN THE TARGET MODEL
# ------------------------------------------------------------

loss_function = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 7

print("\nTraining the target model...")

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        # Move data to CPU/GPU
        images = images.to(device)
        labels = labels.to(device)

        # Remove old gradients
        optimizer.zero_grad()

        # Get model prediction
        predictions = model(images)

        # Calculate loss
        loss = loss_function(
            predictions,
            labels
        )

        # Calculate gradients
        loss.backward()

        # Update model parameters
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"- Loss: {average_loss:.4f}"
    )


# ------------------------------------------------------------
# 6. START MODEL INVERSION ATTACK
# ------------------------------------------------------------

print("\n====================================")
print("STARTING MODEL INVERSION ATTACK")
print("====================================")


# Fashion-MNIST class 9 = Ankle Boot
target_class = 9

model.eval()

print("\nTarget class: 9 (Ankle Boot)")


# ------------------------------------------------------------
# 7. CREATE A RANDOM IMAGE
# ------------------------------------------------------------

# Start with random noise
inverted_image = torch.randn(
    (1, 1, 28, 28),
    device=device,
    requires_grad=True
)

print("Random image created.")


# ------------------------------------------------------------
# 8. OPTIMIZE THE IMAGE
# ------------------------------------------------------------

# IMPORTANT:
# We are changing the IMAGE, not the model.

image_optimizer = optim.Adam(
    [inverted_image],
    lr=0.1
)

number_of_steps = 1500

print("\nOptimizing the image...")

for step in range(number_of_steps):

    # Remove previous gradients
    image_optimizer.zero_grad()

    # Send image through the trained model
    output = model(inverted_image)

    # We want the model to give a high score
    # to the target class (Ankle Boot).
    loss = -output[0, target_class]

    # Calculate gradient with respect to the image
    loss.backward()

    # Modify the image
    image_optimizer.step()

    # Display progress
    if step % 300 == 0:

        print(
            f"Step {step:4d} | "
            f"Loss: {loss.item():.4f}"
        )


print("\nModel inversion completed!")


# ------------------------------------------------------------
# 9. PREPARE THE RECONSTRUCTED IMAGE
# ------------------------------------------------------------

# Detach image from PyTorch computation
image = inverted_image.detach()

# Move image to CPU
image = image.cpu()

# Convert to NumPy
image = image.numpy()

# Remove unnecessary dimensions
image = image.squeeze()

# Normalize image values between 0 and 1
image = (
    image - image.min()
) / (
    image.max() - image.min()
)


# ------------------------------------------------------------
# 10. DISPLAY THE RECONSTRUCTED IMAGE
# ------------------------------------------------------------

plt.figure(figsize=(4, 4))

plt.imshow(
    image,
    cmap="gray"
)

plt.title(
    "Model Inversion: Class 9 (Ankle Boot)"
)

plt.axis("off")

plt.show()

print("\nProgram finished.")